# Python for AI — Class 9
### Generators

Recursion and decorators (Class 8) changed how a function runs. A **generator** changes how it hands back results: one value at a time, only when asked, using `yield` instead of `return`.

It comes with real business examples — invoice numbers, batching — and the day ends with a small shop program that ties generators, decorators and everything so far together, plus a practice set.

**How to use this notebook:** run each cell with the play button, or `Shift + Enter`.

Cells marked **BREAKS ON PURPOSE** are *supposed* to show a red error. Cells marked **WRONG ON PURPOSE** run fine and give the *wrong answer* — those are the dangerous ones. Don't fix either before class; that is the lesson.

# 1. Generators — producing values one at a time

A normal function builds its whole answer and hands it back all at once with `return`. A **generator** hands values out **one at a time**, only when asked, using `yield` instead of `return`.

In [2]:
def count_up_to(n):
    i = 1
    while i <= n:
        yield i          # hand out one value, then pause right here
        i += 1
        # yield i    

gen_func = count_up_to(5)
for number in gen_func:
    # if (number == 2):
    #     gen_func.close()
    print(number)

1
2
3
4
5


`yield` is like a `return` that doesn't end the function — it hands out a value and **pauses**. Next time a value is asked for, the function resumes exactly where it left off, with all its variables intact.

Calling a generator function doesn't run its body at all. It gives you a generator object, and `next()` pulls one value out at a time:

In [ ]:
gen = count_up_to(3)
print(gen)            # a generator object - no code inside has run yet
print(next(gen))      # runs until the first yield
print(next(gen))      # resumes, runs to the next yield
print(next(gen))

<generator object count_up_to at 0x10bdc6810>
1
Computeed total of any XYZ...
2


In [9]:
# BREAKS ON PURPOSE - the generator has nothing left to give
print(next(gen))

StopIteration: 

**StopIteration** — the generator ran out of values. A `for` loop quietly watches for this signal and stops, which is why looping over a generator never shows the error.

A generator can also only be walked through **once**:

In [ ]:
# WRONG ON PURPOSE - the second list is empty, because the generator is used up
numbers = count_up_to(3)
print(list(numbers))
print(list(numbers))

[1, 2, 3]
[]


### Why bother? Memory.

A list stores every value at once. A generator only ever holds one value at a time. Swap the square brackets of a list comprehension (Day 3's sneak peek) for round brackets and you get a **generator expression**:

In [1]:
import sys

squares_list = [n * n for n in range(1000000)]     # builds all million values now
squares_gen = (n * n for n in range(1000000))      # builds each one only when asked

print("List:     ", sys.getsizeof(squares_list), "bytes")
print("Generator:", sys.getsizeof(squares_gen), "bytes")

# print(next(squares_gen))    # 0
# print(next(squares_gen))    # 1



List:      8448728 bytes
Generator: 208 bytes


Millions of bytes against a couple of hundred — and the generator stays that size no matter how big the range gets.

### Real use case: sending orders in batches

An online store might need to hand 5,000 orders to a courier's system — but the courier only accepts 100 at a time. AI work looks the same: sending records to a model in chunks rather than all at once. A generator hands out one batch at a time.

In [ ]:
def batches(items, size):
    for start in range(0, len(items), size):
        yield items[start:start + size]

orders = ["A-1001", "A-1002", "A-1003", "A-1004", "A-1005", "A-1006", "A-1007"]

for batch in batches(orders, 3):
    print("Sending to courier:", batch)

Sending to courier: ['A-1001', 'A-1002', 'A-1003']
Sending to courier: ['A-1004', 'A-1005', 'A-1006']
Sending to courier: ['A-1007']


### Real use case: invoice numbers

Every sale needs its own invoice number, forever. Because a generator only runs when asked, it can safely contain a `while True:` loop — it never runs away on its own.

In [6]:
def invoice_numbers():
    number = 1
    while True:                         # infinite ON PURPOSE - safe, it only runs when asked
        yield f"INV-{number:04d}"       # :04d pads the number to four digits: 0001, 0002...
        number += 1

invoices = invoice_numbers()
print(next(invoices))
print(next(invoices))
print(next(invoices))

INV-0001
INV-0002
INV-0003


> **Where you'll meet generators for real:** Python reads huge files line by line this way, without loading the whole file into memory. And when an AI chatbot streams its reply to you word by word in Month 2, the code receiving that reply is looping over values arriving one at a time — the same idea.

---
# 2. Putting it all together: a day at a small shop

Everything from the last two days, working together in one small program. A computer shop's POS till rings up sales, numbers every receipt, and logs each sale automatically. At closing time, it produces the day's P&L.

Look for each tool as you read: a **generator** for receipt numbers, a **decorator** that logs sales, `*args` for any number of items, a **default argument** for the discount, a **lambda** for the report, and **recursion** for the nested expenses.

In [ ]:
catalogue = {
    "Mouse":    {"price": 1500,  "cost": 900},       # what we sell it for, what we paid for it
    "Keyboard": {"price": 3500,  "cost": 2200},
    "Monitor":  {"price": 45000, "cost": 36000},
}

sales_log = []

def receipt_numbers():                               # generator
    n = 1
    while True:
        yield f"R-{n:03d}"
        n += 1

receipts = receipt_numbers()

def record_sale(func):                               # decorator
    def wrapper(*args, **kwargs):
        sale = func(*args, **kwargs)
        sales_log.append(sale)
        return sale
    return wrapper

@record_sale
def checkout(*items, discount=0):                    # *args, plus a default argument
    revenue = 0
    cost = 0
    for item in items:
        revenue += catalogue[item]["price"]
        cost += catalogue[item]["cost"]
    revenue = revenue * (1 - discount / 100)         # discount is a percentage
    return {"receipt": next(receipts), "items": items, "revenue": revenue, "cost": cost}

One new detail in `checkout(*items, discount=0)`: any parameter written **after** `*items` can only be given by name — `discount=10` — because `*items` swallows every plain value you pass. That's exactly what you want here: no one can accidentally pass a discount as if it were an item.

Now the shop opens, and three customers come in:

In [ ]:
print(checkout("Mouse", "Keyboard"))
print(checkout("Monitor", discount=10))          # a regular customer gets 10% off
print(checkout("Mouse", "Mouse", "Mouse"))

> **Run the setup cell again before re-running this one.** `sales_log` lives outside the functions, so running the checkout cell twice records the same three sales twice — a small, real example of why yesterday's scope lesson warned about shared global state.

Closing time. First, the day's sales from biggest to smallest, then the P&L:

- **Revenue** — money taken from customers
- **Cost of goods sold** — what the shop paid for the items it sold
- **Gross profit** — revenue minus cost of goods sold
- **Operating expenses** — the costs of running the shop at all: rent, staff, electricity
- **Net profit** — what's actually left: gross profit minus operating expenses

In [ ]:
print("Sales, biggest first:")
for sale in sorted(sales_log, key=lambda s: s["revenue"], reverse=True):     # lambda
    print(" ", sale["receipt"], sale["revenue"])

shop_expenses = {                                    # nested - recursion handles it
    "rent": 3000,
    "staff": {"cashier": 2500, "cleaner": 800},
    "utilities": {"electricity": 900, "internet": 300},
}

def total_cost(item):                                # recursion
    if type(item) != dict:
        return item
    amount = 0
    for child in item.values():
        amount += total_cost(child)
    return amount

revenue = 0
cost_of_goods = 0
for sale in sales_log:
    revenue += sale["revenue"]
    cost_of_goods += sale["cost"]

gross_profit = revenue - cost_of_goods
operating_expenses = total_cost(shop_expenses)
net_profit = gross_profit - operating_expenses

print()
print("--- P&L for today ---")
print("Revenue:           ", revenue)
print("Cost of goods sold:", cost_of_goods)
print("Gross profit:      ", gross_profit)
print("Operating expenses:", operating_expenses)
print("Net profit:        ", net_profit)

Rs 50,000 came through the till, but the shop only kept Rs 700 of it. Most of the money went on buying the stock it sold — and almost all of that came from the Monitor sale, which had a thin margin *and* a discount. That's the kind of question a P&L exists to answer, and it took a handful of small functions to produce one.

---
# 3. Practice set: two interview classics

### A calculator function

In [ ]:
def calculate(a, b, operation):
    if operation == "+":
        return a + b
    elif operation == "-":
        return a - b
    elif operation == "*":
        return a * b
    elif operation == "/":
        if b == 0:
            return "Cannot divide by zero"
        return a / b
    else:
        return "Unknown operation"

print(calculate(10, 5, "+"))
print(calculate(10, 5, "/"))
print(calculate(10, 0, "/"))
print(calculate(10, 5, "^"))

### A prime checker

A **prime number** is an integer greater than `1` that has exactly two factors: `1` and itself.

The `is_prime(n)` function checks this step by step:

- If `n < 2`, it returns `False` because `0` and `1` are not prime.
- It tests every number from `2` to `n - 1`.
- If `n % i == 0`, then `n` divides evenly by another number, so it is not prime.
- If no divisor is found, it returns `True`.

For example, `7` is prime because it is not evenly divisible by `2`, `3`, `4`, `5`, or `6`. However, `8` is not prime because `8 % 2 == 0`.

The loop checks numbers from `1` through `19` and prints only those for which `is_prime(number)` returns `True`.

In [ ]:
def is_prime(n):
    if n < 2:
        return False
    for i in range(2, n):
        if n % i == 0:
            return False
    return True

for number in range(1, 20):
    if is_prime(number):
        print(number, end=" ")

The moment a factor turns up, `return False` leaves the function immediately — no need to keep checking. Same "stop as soon as you know the answer" instinct as `break` on Day 3, just via `return` instead.

> This checks every number up to `n - 1`. A faster version only needs to check up to the square root of `n` — not worth the complexity at these sizes, but worth knowing it exists once your numbers get large.

---
# 4. Your turn

**1.** Given `[("Ali", 22), ("Zara", 19), ("Hamza", 25)]`, sort it by age using `.sort()` and a `lambda`, then print it.

**2.** Write a recursive function `count_letters(word)` that returns how many letters a word has, without using `len()`. What is your base case?

**3.** Write `sum_digits(n)` recursively so that `sum_digits(1234)` returns `10`. (Hint: `n % 10` gives the last digit, `n // 10` drops it — you already have this one above, but write it yourself before checking.)

**4.** Write `is_prime(n)`, then use it to print every prime number between 1 and 30.

**5.** Write a decorator `shout` that makes any function which returns text return it in UPPERCASE instead. Test it on a function `greet(name)` that returns `f"hello {name}"`.

**6.** Write a generator `evens(limit)` that yields the even numbers from 2 up to `limit`. Loop over `evens(10)` and print each one.

In [ ]:
# Your practice space

---
### Today you learned

- `lambda parameters: expression` is a one-line, unnamed function — mainly useful as a throwaway `key=` argument for `sorted()`, `.sort()`, `max()` and `min()`
- The `key=` lambda can **calculate** something, like `price * units_sold`, not just pick out a field
- Recursion needs a **base case** and a **recursive case** that moves toward it; without a base case you get `RecursionError`
- Recursion is the natural tool for nested data — folders, product categories, a P&L's expense categories
- A **decorator** takes a function and returns a wrapped version with extra behaviour — logging, timing, permission checks; `@name` above a `def` is the shortcut
- A **generator** uses `yield` to hand out values one at a time, only when asked — it saves memory, handles batches, and can safely run forever
- Any parameter written after `*args` can only be passed by name

**Next:** list and dictionary comprehensions, then `map()`, `filter()` and modules.